# 检查真实模型的独立 Engine 启动 seed
同 prompt / seed 重启 Engine 三次比较文本，不同 seed 各运行一次。
该测试不验收持久 Engine 的逐请求 seed，也不验收 hidden/latent 轨迹对齐。


In [1]:
!pip install -q uv

In [2]:
%%bash

uv python install 3.11
uv venv --python 3.11 --seed /kaggle/working/latent_sglang

Python 3.11 is already installed
Using CPython 3.11.17
Creating virtual environment with seed packages at: latent_sglang
error: Failed to create virtual environment
  cause: A virtual environment already exists at: latent_sglang

hint: Use the `--clear` flag or set `UV_VENV_CLEAR=1` to replace the existing virtual environment


CalledProcessError: Command 'b'\nuv python install 3.11\nuv venv --python 3.11 --seed /kaggle/working/latent_sglang\n'' returned non-zero exit status 2.

In [3]:
%%bash

/kaggle/working/latent_sglang/bin/python --version

Python 3.11.17


In [4]:
%%bash
git clone https://github.com/fengjixing-stupid/ECNU-Adaptive-Noise-Latent-GRPO.git
cd /kaggle/working/ECNU-Adaptive-Noise-Latent-GRPO/latent_grpo_minfix/Latent-GRPO-final
git clone https://github.com/fengjixing-stupid/Latent-GRPO.git

fatal: destination path 'ECNU-Adaptive-Noise-Latent-GRPO' already exists and is not an empty directory.
fatal: destination path 'Latent-GRPO' already exists and is not an empty directory.


CalledProcessError: Command 'b'git clone https://github.com/fengjixing-stupid/ECNU-Adaptive-Noise-Latent-GRPO.git\ncd /kaggle/working/ECNU-Adaptive-Noise-Latent-GRPO/latent_grpo_minfix/Latent-GRPO-final\ngit clone https://github.com/fengjixing-stupid/Latent-GRPO.git\n'' returned non-zero exit status 128.

In [5]:
%%bash
cd /kaggle/working/ECNU-Adaptive-Noise-Latent-GRPO/latent_grpo_minfix/Latent-GRPO-final/Latent-GRPO/Latent-GRPO/sglang_latent_reasoning_pkg
/kaggle/working/latent_sglang/bin/python -m pip install -U pip

/kaggle/working/latent_sglang/bin/python -m pip install -e "python[all]"

Obtaining file:///kaggle/working/ECNU-Adaptive-Noise-Latent-GRPO/latent_grpo_minfix/Latent-GRPO-final/Latent-GRPO/Latent-GRPO/sglang_latent_reasoning_pkg/python
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for sglang (pyproject.toml): started
  Building editable for sglang (pyproject.toml): finished with status 'done'
  Created wheel for sglang: filename=sglang-0.4.6.post1-0.editable-py3-none-any.whl size=4271 sha256=a8e30d082abd31835a7047e18ed1f0ccacb8d5af95c7fc9ec1f4ac86729923a0
  Stored in directory: /tmp/pip-ephem-wheel

In [6]:
%%bash

/kaggle/working/latent_sglang/bin/python - <<'PY'
import sys
import sglang

print("Python:", sys.version)
print("SGLang:", sglang.__version__)
PY

Python: 3.11.17 (main, Oct  3 2026, 01:05:36) [Clang 22.1.3 ]
SGLang: 0.4.6.post1


In [7]:
%%bash

PY=/kaggle/working/latent_sglang/bin/python

$PY -m pip uninstall -y torchao compressed-tensors

$PY -m pip install \
    torchao==0.9.0 \
    compressed-tensors==0.11.0

Found existing installation: torchao 0.9.0
Uninstalling torchao-0.9.0:
  Successfully uninstalled torchao-0.9.0
Found existing installation: compressed-tensors 0.11.0
Uninstalling compressed-tensors-0.11.0:
  Successfully uninstalled compressed-tensors-0.11.0
  Using cached torchao-0.9.0-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.manylinux_2_28_x86_64.whl.metadata (14 kB)
  Using cached compressed_tensors-0.11.0-py3-none-any.whl.metadata (7.0 kB)
Using cached torchao-0.9.0-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.manylinux_2_28_x86_64.whl (5.7 MB)
Using cached compressed_tensors-0.11.0-py3-none-any.whl (179 kB)



In [8]:
%%bash

/kaggle/working/latent_sglang/bin/python - <<'PY'
import torch
import torchao
import transformers
import compressed_tensors
import sglang

from importlib.metadata import version

print("torch:", torch.__version__)
print("torchao:", version("torchao"))
print("transformers:", transformers.__version__)
print("compressed-tensors:", version("compressed-tensors"))
print("sglang:", sglang.__version__)

from sglang.srt.entrypoints.engine import Engine
print("Engine import OK")
PY

torch: 2.6.0+cu124
torchao: 0.9.0
transformers: 4.51.1
compressed-tensors: 0.11.0
sglang: 0.4.6.post1
Engine import OK


/kaggle/working/latent_sglang/lib/python3.11/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/kaggle/working/latent_sglang/lib/python3.11/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [ ]:
%%bash

cat > /kaggle/working/test_sglang_seed.py <<'PY'
import asyncio
import hashlib
from pathlib import Path

import torch
import sglang as sgl
from transformers import AutoTokenizer


MODEL_PATH = "/kaggle/input/models/fengjixing/llama3-2-1b-instruct-latent-sft-top10/pytorch/latent-sft/1/LLaMA3.2-1B-Instruct-Latent-SFT-Top10"

PROMPT = (
    "Write one short imaginative nonsense sentence using unusual word choices. "
    "Do not explain anything; output only the sentence."
)

SAME_SEED = 123456
DIFFERENT_SEEDS = [111111, 222222, 333333]

TEMPERATURE = 1.0
MAX_NEW_TOKENS = 64


def sample_once(prompt: str, seed: int) -> str:
    print(f"\n[seed={seed}] loading tokenizer...", flush=True)

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_PATH,
        local_files_only=True,
        trust_remote_code=True,
    )

    text = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True,
    )

    if not text.rstrip().endswith("<think>"):
        text += "<think>"

    input_ids = tokenizer.encode(
        text,
        add_special_tokens=False,
    )

    end_ids = tokenizer.encode(
        "</think>",
        add_special_tokens=False,
    )

    print(
        f"[seed={seed}] creating SGLang Engine...",
        flush=True,
    )

    engine = sgl.Engine(
        model_path=MODEL_PATH,
        trust_remote_code=True,
        dtype="float16",
        kv_cache_dtype="auto",
        tp_size=1,
        base_gpu_id=0,
        random_seed=seed,
        enable_latent=True,
        latent_end_token_id=end_ids[0],
        disable_cuda_graph=True,
        disable_overlap_schedule=True,
        mem_fraction_static=0.90,
        sampling_backend="flashinfer",
        max_running_requests=1,
        log_level="info",
        skip_tokenizer_init=True,
        max_topk=10,
    )
    
    # 重要：
    # sgl.Engine() 首次调用会加载 SGLang runtime，并安装 uvloop policy。
    # 因此必须在 Engine 创建完成后，再创建当前 event loop。
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    
    print(
        f"[seed={seed}] Engine READY, starting generation...",
        flush=True,
    )
    
    try:
        output = engine.generate(
            input_ids=input_ids,
            sampling_params={
                "temperature": TEMPERATURE,
                "top_p": 0.95,
                "max_new_tokens": MAX_NEW_TOKENS,
                "gumbel_softmax_temperature": 1.0,
                "noise_scale": 1.0,
                "add_noise_gumbel_softmax": True,
                "use_one_sided_gumbel_noise": False,
            },
        )
    
        print(
            f"[seed={seed}] generation finished",
            flush=True,
        )
    
        return tokenizer.decode(
            output["output_ids"],
            skip_special_tokens=False,
        )
    
    finally:
        print(
            f"[seed={seed}] shutting down Engine",
            flush=True,
        )
    
        engine.shutdown()
    
        loop.close()
        asyncio.set_event_loop(None)


def short_hash(text):
    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()[:12]


def main():

    print("Python / CUDA check")
    print("torch:", torch.__version__)
    print("sglang:", sgl.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("GPU count:", torch.cuda.device_count())

    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}:", torch.cuda.get_device_name(i))

    print("\n先只做一次测试，确认 Engine 能正常启动。")
    print("=" * 80)

    text = sample_once(PROMPT, SAME_SEED)

    print("\nRESULT")
    print("hash:", short_hash(text))
    print(text)

    print("\n不同 seed 检查：各生成一次")
    print("=" * 80)
    different_outputs = []
    for seed in DIFFERENT_SEEDS:
        text = sample_once(PROMPT, seed)
        different_outputs.append(text)
        print(f"[different] seed={seed} hash={short_hash(text)}")
        print(repr(text))

    if len(set(different_outputs)) > 1:
        print("Different-seed check: PASS (outputs differ)")
    else:
        print("Different-seed check: WARN (outputs are identical; seed effectiveness is inconclusive)")


if __name__ == "__main__":
    main()
PY


PYTHONUNBUFFERED=1 \
/kaggle/working/latent_sglang/bin/python \
    /kaggle/working/test_sglang_seed.py

In [ ]:
%%bash
PYTHONUNBUFFERED=1 \
/kaggle/working/latent_sglang/bin/python \
/kaggle/working/test_sglang_seed.py

## 结果范围
PASS 表示这六次独立 Engine 运行中，同 seed 的解码文本一致，且不同 seed 至少产生两种文本。
文本结果不能替代 latent 轨迹、worker RNG 消费和持久 Engine 请求级 seed 的验收。
FAIL 表示同 seed 文本不一致；WARN 表示不同 seed 文本一致，不能仅据此判断 seed 无效。
出现环境、依赖、设备或 OOM 错误时停止并保留报错，不自动降级重试。
